# Testing Manager

This notebook is a step-by-step walkthrough for the exact workflow you want:

1. generate an experiment plan JSON
2. load the plan
3. preview it before touching hardware
4. execute it in the lab
5. inspect the log file afterwards

It uses a dummy `VB-1-13`-style experiment by default:
- one source vial
- one-component slug
- 100 uL slug volume
- repeated identical slugs
- needle wash intended between repeats later

Read the comments in each code cell before running it.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

# This makes the notebook work whether VS Code starts the kernel in
# the repo root or inside the Notebooks folder.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Notebooks" else NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Core.experiment_builder import ExperimentBuilder
from Core.experiment_context import ExperimentManager

# pandas is optional here. If it is not installed, the notebook still works.
try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

print(f"Project root: {PROJECT_ROOT}")
print(f"pandas available: {pd is not None}")

## 1. Define the experiment inputs

Edit only the values in the next cell if you want to change the dummy experiment.

This is intentionally shaped to feel like `VB-1-13`:
- all slugs are identical
- each slug uses one vial only
- each slug is 100 uL
- repeat count is easy to change

In [ ]:
# Change these values to define a new experiment.
# For tomorrow, this is the main cell you will edit.

experiment_id = "vb_1_13_dummy"
description = "Dummy notebook version of VB-1-13 for manager testing"

# Source vial for the slug component.
source_module = "rack1"
source_vial = 1

# Slug definition.
n_repeats = 15
slug_volume_uL = 100
air_gap_between_uL = 5
dispense_rate_mL_min = 0.5

# This is not yet enforced automatically during execution, but it is useful
# to store now so the plan clearly states the intended run policy.
wash_policy = "needle"

# Optional free-text notes for your future self.
notes = "Benzyl alcohol leaching characterisation dummy plan"

## 2. Build the tabular rows

Each row represents one component pickup. Since this dummy experiment has one-component slugs, there is one row per slug.

Later, for true mixtures, the same `slug_id` would appear on multiple rows.

In [ ]:
rows = []

for i in range(1, n_repeats + 1):
    rows.append(
        {
            "slug_id": f"{experiment_id}_slug_{i:02d}",
            "slug_order": i,
            "component_order": 1,
            "module": source_module,
            "vial": source_vial,
            "volume_uL": slug_volume_uL,
        }
    )

# This plain Python list is enough. pandas is optional.
rows[:3]

In [ ]:
if pd is None:
    print("pandas is not installed in this kernel. That is fine.")
    print(f"Number of rows prepared: {len(rows)}")
else:
    display(pd.DataFrame(rows))

## 3. Generate the experiment folder

This writes `plan.json` and `log.json` into `Experiments/<experiment_id>/`.

`overwrite=True` means you can rerun this cell after editing the inputs above.

In [ ]:
builder = ExperimentBuilder()

result = builder.build_and_create(
    experiment_id=experiment_id,
    rows=rows,
    description=description,
    global_conditions={
        "slug_volume_uL": slug_volume_uL,
        "air_gap_between_uL": air_gap_between_uL,
        "dispense_rate_mL_min": dispense_rate_mL_min,
        "wash_policy": wash_policy,
        "notes": notes,
    },
    overwrite=True,
)

print(f"Plan written to: {result['plan_path']}")
print(f"Log written to:  {result['log_path']}")
result["summary"][:3]

In [ ]:
# Open the generated plan so you can inspect exactly what will be loaded.
with open(result["plan_path"], "r") as f:
    plan = json.load(f)

plan

## 4. Load and preview with `ExperimentManager`

This section is home-safe. No instruments are touched here.

In [ ]:
# This loads the generated experiment folder and prepares the manager state.
manager = ExperimentManager()
manager.load_experiment(experiment_id)
manager.status()

In [ ]:
# Preview gives you a compact summary of the whole planned run.
# This is the cell I would always run before executing in the lab.
preview_rows = manager.preview()
preview_rows[:5]

In [ ]:
# Optional: pull just the next slug dict so you can inspect its exact shape.
# If you run this, the manager index advances by one.
slug = manager.get_next_slug()
slug

In [ ]:
# Optional: show what remains after get_next_slug() has been called.
manager.remaining_slugs()[:3]

## 5. Lab execution pattern

Only run this section after your real hardware setup notebook cells have created a live `rsg` object.

Important:
- if you used `get_next_slug()` above for inspection, restart from a fresh manager before executing
- that avoids accidentally skipping slug 1

In [ ]:
# LAB ONLY
# Use this first. It is the safest first test tomorrow.
# It makes exactly one slug from the plan.

# manager = ExperimentManager()
# manager.load_experiment(experiment_id)
# manager.begin_run()
# slug = manager.get_next_slug()
# result = rsg.create_slug(slug)
# print(slug)
# print(result)

In [ ]:
# LAB ONLY
# Use this after the single-slug test behaves correctly.
# This runs all remaining slugs in sequence using the plan defaults.

# manager = ExperimentManager()
# manager.load_experiment(experiment_id)
# manager.begin_run()
# results = manager.run_remaining_slugs(rsg)
# results

## 6. Inspect the log file

Before execution, the log should mostly be empty apart from creation metadata.
After execution, this is where you should see `begin_run`, `slug_created`, and `all_slugs_completed` events.

In [ ]:
with open(result["log_path"], "r") as f:
    log_data = json.load(f)

log_data

In [ ]:
# This gives a quick count of recorded events.
len(log_data.get("events", []))

## 7. How to adapt this to the real `VB-1-13`

Tomorrow, you should be able to do the following with minimal edits:

1. set `experiment_id` to the real experiment name
2. set `source_module` and `source_vial` to the real stock location
3. set `n_repeats` to the real repeat count
4. rebuild the experiment folder
5. run the preview cell
6. execute one slug first
7. only then run the full sequence